# Consistent Stock Selection Portfolio

## Goal

This notebook help evaluate if a stock that the model consistently selects performs better then the S&P 500.

While the Random Forest model had performance better than random, it did not have enough to have returns better than S&P 500. This notebook focuses on stocks with repeated high probabilities in order to find more consistently good stocks.

## 1. Load Model Predictions

The classifier output contains:

* Ticker
* Date of Window
* Model Probabilities
* Model Predictions
* Data features
* Weather the stock beat the S&P 500

In [ ]:
# Importing File
import pandas as pd
import numpy as np
import yfinance as yf

# Parameters
SCORED_FILE = "../logs/classification/test_scored_gen1.parquet"

TOP_N = 10
MIN_SELECTIONS = 10
PORTFOLIO_SIZE = 20

df = pd.read_parquet( SCORED_FILE )

df = df.reset_index()

# Test print of (Rows, Columns)
df.shape

(147236, 122)


,index,date,ticker,price,beta_1y,alpha_1y,sector,quote_type,log_market_cap,pe,...,risk_adjusted_5y,sector_is_strong,sector_is_trending,sector_high_breadth,quality_score,beats_market_5y,logreg_probability,logreg_prediction,rf_probability,rf_prediction
0,0,2020-07-06,EGHT,15.98,1.640449,-0.033117,Technology,EQUITY,21.541141,1480.114852,...,0.076579,True,False,False,0,0,0.331555,0,0.340725,0
1,1,2020-07-13,EGHT,15.52,1.631050,-0.034542,Technology,EQUITY,21.511933,1437.508375,...,0.101426,True,False,False,0,0,0.319037,0,0.335861,0
2,2,2020-07-20,EGHT,16.93,1.625617,0.054172,Technology,EQUITY,21.598891,1568.106734,...,0.117621,True,False,True,0,0,0.359690,0,0.346215,0
3,3,2020-07-27,EGHT,16.42,1.624197,0.016425,Technology,EQUITY,21.568304,1520.869003,...,0.104128,True,False,False,0,0,0.331006,0,0.337873,0
4,4,2020-08-03,EGHT,16.25,1.629430,0.013561,Technology,EQUITY,21.557896,1505.123093,...,0.080650,True,False,True,0,0,0.345362,0,0.342717,0


## 2. Identify Top Model Picks

For each predition date, the stocks with the highest model confience are selected. These represent the stocks the model would have purchased at each time window.

In [ ]:
# Get top N highest RF probabilities
top_picks = (
    df.sort_values(
        ["date", "rf_probability"],
        ascending=[True, False]
    )
    .groupby("date")
    .head(TOP_N)
)

top_picks.head()

,index,date,ticker,price,beta_1y,alpha_1y,sector,quote_type,log_market_cap,pe,...,risk_adjusted_5y,sector_is_strong,sector_is_trending,sector_high_breadth,quality_score,beats_market_5y,logreg_probability,logreg_prediction,rf_probability,rf_prediction
115300,115300,2020-07-06,MKSI,111.151100,1.794942,0.010785,Technology,EQUITY,22.739182,21.972115,...,0.785387,True,False,False,3,0,0.357134,0,0.467851,0
92337,92337,2020-07-06,AVGO,28.075953,1.615283,0.001960,Technology,EQUITY,25.617918,5.077504,...,0.579143,True,False,False,3,1,0.352905,0,0.460608,0
38121,38121,2020-07-06,AMAT,60.241611,1.729459,0.031159,Technology,EQUITY,24.590906,5.354276,...,0.908165,True,False,False,3,1,0.354921,0,0.456200,0
84448,84448,2020-07-06,FICO,416.850006,1.766286,-0.013949,Technology,EQUITY,22.991997,12.458078,...,1.306724,True,False,False,3,1,0.381020,0,0.454755,0
77692,77692,2020-07-06,KLAC,19.084976,1.631847,0.027378,Technology,EQUITY,23.939347,4.974626,...,1.185625,True,False,False,3,1,0.350392,0,0.453493,0


## 3. Measure Stock Consistency

How often a stock is selected is then counted. Maximum for this is 51, as there are 51 possible time windows.

Stocks selected repeatedly represent stronger model conviction.

In [ ]:
# Sort stocks by number of times selected
consistent = (
    top_picks
    .groupby("ticker")
    .agg(
        times_selected=("ticker", "count"),
        avg_probability=("rf_probability", "mean"),
        median_probability=("rf_probability", "median"),
        avg_future_return=("future_excess_5y", "mean"),
        median_future_return=("future_excess_5y", "median"),
        beat_rate=("beats_market_5y", "mean")
    )
    .sort_values(
        "times_selected",
        ascending=False
    )
)

consistent.head(50)

,times_selected,avg_probability,median_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,,
AVGO,31,0.463374,0.467906,7.782043,7.836940,1.000000
TXN,24,0.446163,0.445278,-0.410822,-0.484456,0.083333
ACN,23,0.445970,0.443433,-1.012741,-0.981327,0.000000
KLAC,19,0.467379,0.469013,3.033359,2.775119,1.000000
DIOD,18,0.474531,0.477672,-1.200443,-1.228786,0.000000
ADI,18,0.459058,0.461986,0.176064,0.092871,0.833333
CP,17,0.431363,0.432161,-0.742496,-0.694214,0.000000
FIS,15,0.458876,0.462578,-1.524785,-1.491656,0.000000
MSI,15,0.451964,0.450089,0.448461,0.266082,1.000000


## 4. Filter stocks below threshold

A list of stocks have have been selected at or above the threshold are selected. Stocks only selected by the algorithm a few times are discarded as inconsistent.

In [ ]:
# Filter to stocks have have been selected at, or above, limit
consistent_stocks = (
    consistent[
        consistent["times_selected"] >= MIN_SELECTIONS
    ]
)

consistent_stocks

,times_selected,avg_probability,median_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,,
AVGO,31,0.463374,0.467906,7.782043,7.836940,1.000000
TXN,24,0.446163,0.445278,-0.410822,-0.484456,0.083333
ACN,23,0.445970,0.443433,-1.012741,-0.981327,0.000000
KLAC,19,0.467379,0.469013,3.033359,2.775119,1.000000
DIOD,18,0.474531,0.477672,-1.200443,-1.228786,0.000000
ADI,18,0.459058,0.461986,0.176064,0.092871,0.833333
CP,17,0.431363,0.432161,-0.742496,-0.694214,0.000000
FIS,15,0.458876,0.462578,-1.524785,-1.491656,0.000000
MSI,15,0.451964,0.450089,0.448461,0.266082,1.000000


## 5. Limit number of stocks selected

The number of consistently selected stocks are capped to ensure both selection of the best results as well as keeping a simple portfolio.

In [ ]:
# Filter stocks down to maximum number
portfolio = (
    consistent_stocks
    .sort_values(
        [
            "times_selected",
            "avg_probability"
        ],
        ascending=False
    )
    .head(PORTFOLIO_SIZE)
)

portfolio[
    [
        "times_selected",
        "avg_probability",
        "avg_future_return",
        "median_future_return",
        "beat_rate"
    ]
]

,times_selected,avg_probability,avg_future_return,median_future_return,beat_rate
ticker,,,,,
AVGO,31,0.463374,7.782043,7.836940,1.000000
TXN,24,0.446163,-0.410822,-0.484456,0.083333
ACN,23,0.445970,-1.012741,-0.981327,0.000000
KLAC,19,0.467379,3.033359,2.775119,1.000000
DIOD,18,0.474531,-1.200443,-1.228786,0.000000
ADI,18,0.459058,0.176064,0.092871,0.833333
CP,17,0.431363,-0.742496,-0.694214,0.000000
FIS,15,0.458876,-1.524785,-1.491656,0.000000
MSI,15,0.451964,0.448461,0.266082,1.000000


In [ ]:
# Most current window of data
last_date = df["date"].max()

print(last_date)

2021-07-01 00:00:00


## 6. Generate Portfolio List

A list of the top stocks.

This could be expanded to include features of the stocks, either for the most current window or an average over all windows.

In [ ]:
#Tickers for all stocks in portfolio
holdings = portfolio.index.tolist()

holdings

['AVGO',
 'TXN',
 'ACN',
 'KLAC',
 'DIOD',
 'ADI',
 'CP',
 'FIS',
 'MSI',
 'TMTNF',
 'SMTC',
 'BR',
 'FICO',
 'STM',
 'TDY',
 'UNP',
 'TRI',
 'GRMN',
 'WOLTF',
 'ITOCF']

In [49]:
buy_and_hold = df[
    (df["date"] == last_date) &
    (df["ticker"].isin(holdings))
].copy()

buy_and_hold[
    [
        "ticker",
        "rf_probability",
        "future_excess_5y",
        "beats_market_5y"
    ]
]

,ticker,rf_probability,future_excess_5y,beats_market_5y
16471,CP,0.410044,-0.688673,0
29304,ADI,0.422538,0.648811,1
32574,TMTNF,0.405139,0.176961,1
38273,MSI,0.444481,0.178112,1
62077,BR,0.358905,-0.909200,0
70747,ACN,0.435611,-1.383258,0
77742,KLAC,0.357921,7.129102,1
79903,TRI,0.414366,-0.912384,0
84498,FICO,0.386459,0.557984,1
86167,UNP,0.415770,-0.462905,0


In [ ]:
spy = yf.download(
    "SPY",
    start=last_date,
    end=pd.Timestamp(last_date) + pd.DateOffset(years=5),
    auto_adjust=True
)

# Result Summary

The excess returns (return above SPY) and actual returns are both calculated.

For easier understanding, a simulation is also run, based on a $10,000 investment, equally distributed to each stock.

The results suggest that repeated model selection is useful as an investment signal.

In [ ]:
portfolio_return = buy_and_hold["future_ret_5y"].mean()
portfolio_excess = buy_and_hold["future_excess_5y"].mean()

print(
    f"Portfolio excess return: {portfolio_excess:.2%}"
)

print(
    f"Portfolio return: {portfolio_return:.2%}"
)

spy_return = (
    spy["Close"].iloc[-1].item() /
    spy["Close"].iloc[0].item()
    - 1
)

initial_investment = 10000

spy_value = initial_investment * (1 + spy_return)

ending_value = (
    initial_investment *
    (1 + portfolio_return)
)

print(
    "\n",
    "---Portfolio simulation ---",
)

print(
    f"Starting value: ${initial_investment:,.2f}"
)

print(
    f"SPY Ending value: ${spy_value:,.2f}"
)

print(
    f"Model Ending value: ${ending_value:,.2f}"
)

return_difference = ( ending_value - spy_value ) / spy_value

print(
    f"Different compared to SPY: {return_difference:+.2%}"
)

Portfolio excess return: 33.60%
Portfolio return: 118.98%

 ---Portfolio simulation ---
Starting value: $10,000.00
SPY Ending value: $18,562.83
Model Ending value: $21,897.90
Different compared to SPY: +17.97%
